# 23d · Long / flat Double DQN on the enriched factors, checkpoint chosen by timing skill (E23 step 5)

E23c (Double DQN on the real factors, a decision every 60 bars, checkpoint = best V1 Sharpe) failed on V2 in all
6 runs (enriched: V2 Sharpe -1.58, current: -3.42; buy and hold +3.68). The cause was the choice, not only the agent:
V1 is a falling market (buy and hold Sharpe -1.43) and V2 a rising one, so the best-V1-Sharpe checkpoint was always
the mostly-short one, and short exposure in V2 cost 17-26 points. Its **timing** was real, though: the enriched
checkpoints gained +10 to +23 points from timing on V2 (daily IC up to 0.23).

Two changes, everything else as E23c:

* **Long or flat only** (`allow_short=False`): the agent can no longer bet on a falling market; it can only choose
  when to hold BTC.
* **Checkpoint chosen by skill on V1**, not Sharpe: skill % = timing % - costs %, what a checkpoint earns beyond its
  average exposure (diagnose.attribution), so the direction of V1 cannot decide the choice. The run stays in cash
  unless V1 skill is positive. The best-V1-Sharpe checkpoint is still reported next to it.

* **Factors:** enriched (E23b), cross-fitted train + validation. **Agent:** E16 hyperparameters, state A, a decision
  every 60 bars, **5 seeds** x 1M steps. **References on V2:** buy and hold, long / flat threshold rules chosen on
  train and on V1, and E23c's enriched runs.

**Test and stress_bear are not loaded.** T4 GPU, 5 runs of about 15 min (about 1.3 h); runs resume after a
disconnect. Writes `thesis_rl/initiation/e23d_long_flat_g60/` and `report/e23d_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "23d_long_flat_skill")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, halves as HV, walkforward as WF
from rltrade.env import EnvConfig, Market

INIT = L.root / "initiation"
OUT, REP = INIT / "e23d_long_flat_g60", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
FACTORS = INIT / "e23_factors" / "crossfit_mean3"
assert (FACTORS / "factors_train.npy").exists() and (FACTORS / "factors_val.npy").exists(), f"missing {FACTORS} (run 23b)"
DATA, CUTS = HV.write_halves(PREP, FACTORS, LOCAL / "e23d" / "enriched")

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
AGENT_HP, ENV_OVER = hp["agent_hp"], {**hp["env"], "step_gap": 60, "allow_short": False}
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
CFG = EnvConfig(state="A", **ENV_OVER)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1, 2, 3, 4), 1_000_000, 100_000, 5
def run_name(seed): return f"enriched_longflat_g60_s{seed}"
print("agent:", AGENT_HP, ENV_OVER, "| device", DEV, "| V1 / V2 rows:", CUTS["v1"]["rows"], CUTS["v2"]["rows"])

## 1 · References: buy and hold and long / flat threshold rules (chosen on train, and chosen on V1)

In [ ]:
REF = HV.reference(DATA, CFG, DEV)
BH = {s: m["sharpe"] for s, m in REF.items()}
BH_RET = {s: m["cum_return_pct"] for s, m in REF.items()}
fac = np.load(DATA / "factors_train.npy")
grid = FR.make_grid(FR.signals(fac))
res = {s: FR.run_rules(grid, Market(DATA, DATA, s, CFG, DEV), CFG) for s in ("train", "v1", "v2")}   # short signals -> flat
rule_rows = []
for chosen_on in ("train", "v1"):
    i = WF.select_rule(res[chosen_on], MIN_TRADES); g = grid[i]
    rule_rows.append({"rule chosen on": chosen_on, "rule": f"{g['signal']} q{g['q']} {g['mode']} hold>={g['min_hold']}",
                      **{f"{s} Sharpe": res[s]["sharpe"][i] for s in ("train", "v1", "v2")},
                      "V2 return %": res["v2"]["cum_return_pct"][i], "V2 max DD %": res["v2"]["max_drawdown_pct"][i],
                      "V2 trades": int(res["v2"]["num_trades"][i]), "V2 in market %": res["v2"]["exposure_pct"][i]})
rule_df = pd.DataFrame(rule_rows)
rule_df.to_csv(OUT / "rule_reference.csv", index=False)
print("buy and hold (exact, 60-bar decisions): " + ", ".join(f"{s} Sharpe {BH[s]:+.2f} ({BH_RET[s]:+.1f} %)" for s in HV.SPLITS))
print("\nlong / flat threshold rules (decision every 60 bars):\n" + rule_df.round(3).to_string(index=False))

## 2 · Train (about 15 min per run; resumes)

In [ ]:
t0 = time.time()
for seed in SEEDS:
    HV.train_select(run_name(seed), DATA, OUT, seed, AGENT_HP, ENV_OVER, TOTAL_STEPS, EVAL_EVERY, device=DEV,
                    min_trades=MIN_TRADES, select_by="skill")
print(f"training done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results: the checkpoint chosen on V1 judged on V2 (by skill, and by Sharpe for comparison)

In [ ]:
RES = {s: json.loads((OUT / "runs" / run_name(s) / "result.json").read_text()) for s in SEEDS}
def sel_cols(sel, tag):
    if not sel:
        return {f"{tag} step": None, f"{tag} stayed flat": True, f"{tag} V2 Sharpe": 0.0, f"{tag} V2 return %": 0.0}
    v1, v2, flat = sel["v1"], sel["v2"], sel["fallback"]
    return {f"{tag} step": sel["steps"], f"{tag} V1 Sharpe": v1["sharpe"], f"{tag} V1 skill %": v1["skill_pct"],
            f"{tag} ckpt V2 Sharpe": v2["sharpe"], f"{tag} ckpt V2 return %": v2["cum_return_pct"],
            f"{tag} ckpt V2 skill %": v2["skill_pct"], f"{tag} ckpt V2 in market %": v2["exposure_pct"],
            f"{tag} ckpt V2 daily IC": v2["daily_ic"], f"{tag} stayed flat": flat,
            f"{tag} V2 Sharpe": 0.0 if flat else v2["sharpe"], f"{tag} V2 return %": 0.0 if flat else v2["cum_return_pct"],
            f"{tag} V2 max DD %": 0.0 if flat else v2["max_drawdown_pct"], f"{tag} V2 trades": 0 if flat else v2["num_trades"]}
rows = [{"seed": s, **sel_cols(r["selections"]["skill"], "skill"), **sel_cols(r["selections"]["sharpe"], "sharpe"),
         "final V2 Sharpe": r["final"]["v2"]["sharpe"], "final V2 return %": r["final"]["v2"]["cum_return_pct"]}
        for s, r in RES.items()]
runs = pd.DataFrame(rows)
runs.to_csv(OUT / "runs_summary.csv", index=False)
keys = ["skill V2 Sharpe", "skill V2 return %", "skill ckpt V2 Sharpe", "skill ckpt V2 return %", "skill ckpt V2 skill %",
        "sharpe V2 Sharpe", "sharpe V2 return %", "sharpe ckpt V2 Sharpe", "final V2 Sharpe"]
agg = runs[keys].agg(["mean", "std"]).T.round(2)
e23c = []
for s in (0, 1, 2):
    f = INIT / "e23c_real_g60" / "runs" / f"enriched_g60_s{s}" / "result.json"
    if f.exists():
        sel = json.loads(f.read_text())["selected"]
        e23c.append(0.0 if (not sel or sel["fallback"]) else sel["v2"]["sharpe"])
pd.set_option("display.width", 250)
print("every run (ckpt = the chosen checkpoint on V2; V2 = what is traded after the stay-flat rule):")
print(runs.round(2).to_string(index=False))
print("\nmean and SD over seeds:\n" + agg.to_string())
print(f"\nV2 references: buy and hold Sharpe {BH['v2']:+.2f} ({BH_RET['v2']:+.1f} %)"
      + (f"; E23c enriched (long/short, Sharpe choice) mean V2 Sharpe {np.mean(e23c):+.2f} over {len(e23c)} seeds" if e23c else ""))
print(rule_df[["rule chosen on", "rule", "v2 Sharpe", "V2 return %", "V2 trades", "V2 in market %"]].round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
panels = (("train_sharpe", "train (chunked): Sharpe", "train"), ("v1_skill_pct", "V1 (choice): skill % = timing - costs", None),
          ("v2_sharpe", "V2 (judged): Sharpe", "v2"), ("v2_skill_pct", "V2 (judged): skill %", None))
for ax, (key, title, bh) in zip(axes, panels):
    for s, r in RES.items():
        c = pd.DataFrame(r["curve"])
        ax.plot(c.steps / 1e6, c[key], lw=1.2, alpha=0.8, label=f"seed {s}")
        sel = r["selections"]["skill"]
        if sel:
            ax.plot(sel["steps"] / 1e6, c.set_index("steps").loc[sel["steps"], key], "o", color=ax.lines[-1].get_color(), ms=6)
    if bh: ax.axhline(BH[bh], color="black", ls="--", lw=1, label="buy and hold")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(title, fontsize=10); ax.set_xlabel("training steps (millions)"); ax.grid(alpha=0.3)
axes[2].legend(fontsize=7); fig.suptitle("long / flat Double DQN, enriched factors (dots = checkpoint chosen by V1 skill)", fontsize=11)
fig.tight_layout(); fig.savefig(REP / "e23d_curves.png", dpi=130); plt.show()

In [ ]:
(REP / "e23d_long_flat_skill.md").write_text(
    "# E23d: long / flat Double DQN on the enriched factors, checkpoint chosen by V1 skill (train + validation only)\n\n"
    f"State A, E16 hyperparameters, decision every 60 bars, no shorts, {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps. "
    f"Checkpoint = best V1 skill (timing % - costs %) with >= {MIN_TRADES} V1 trades, judged once on V2; stay flat unless "
    "V1 skill > 0. The best-V1-Sharpe checkpoint is reported for comparison (stay flat unless it beats cash and buy and hold "
    "on V1). Test and stress_bear not loaded.\n\n"
    f"V1 rows {CUTS['v1']['rows']}, V2 rows {CUTS['v2']['rows']} of validation. Buy and hold: "
    + ", ".join(f"{s} Sharpe {BH[s]:+.2f} ({BH_RET[s]:+.1f} %)" for s in HV.SPLITS)
    + (f". E23c enriched (long/short, Sharpe choice): mean V2 Sharpe {np.mean(e23c):+.2f} over {len(e23c)} seeds" if e23c else "")
    + "\n\n## Mean and SD over seeds\n\n" + agg.reset_index().to_markdown(index=False)
    + "\n\n## Every run\n\n" + runs.round(3).to_markdown(index=False)
    + "\n\n## Long / flat threshold rules\n\n" + rule_df.round(3).to_markdown(index=False)
    + "\n\nFigure: e23d_curves.png\n", encoding="utf-8")
print("wrote", REP / "e23d_long_flat_skill.md")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()